In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from glob import glob

# Aplicar configuraciones de visualización total de Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

from pathlib import Path
import glob

import warnings
warnings.filterwarnings("ignore")
import plotly.express as px
import func_graff as graff

# 1. Insumos

In [3]:
periodo_analisis = "202607"
ruta = r"bd\bd_productividad.xlsx"
ruta_velocdidad = r"bd\bd_velocidad.xlsx"

# 2. Productividad

In [ ]:
df_prod = pd.read_excel(ruta)
df_prod1 = df_prod[df_prod["GOS"].isin(['Fátima L', 'Sofía V', 'Sebastián S', 'Hernán P', 'María R'])][['Tienda', 'Periodo', 'GOS', 'GOR', 'Prod', 'VtaNeta', 'JEq',
       'ProdMeta', 'VtaNetaMeta', 'JEqMeta', 'ProdAA', 'VtaNetaAA', 'JEqAA']]

df_prod1["Periodo"] = df_prod1["Periodo"].astype(int).astype(str)
df_prod2 = df_prod1[df_prod1["Periodo"]==periodo_analisis]
venta = df_prod2["VtaNeta"].sum()
jeq = df_prod2["JEq"].sum()
venta_meta = df_prod2["VtaNetaMeta"].sum()
jeq_meta = df_prod2["JEqMeta"].sum()
venta_aa = df_prod2["VtaNetaAA"].sum()
jeq_aa = df_prod2["JEqAA"].sum()

prod = venta/jeq
prod_meta = venta_meta/jeq_meta
prod_aa = venta_aa/jeq_aa
var_meta = prod/prod_meta
var_aa = prod/prod_aa
df_prod1["año"] = df_prod1["Periodo"].str[0:4]
df_prod1["mes"] = df_prod1["Periodo"].str[-2:]

## 2.1. Historico

In [ ]:

dffx = df_prod1[(df_prod1["año"]=="2026")&(df_prod1["Periodo"]<=periodo_analisis)].groupby(["mes"]).agg({"VtaNeta":"sum", "JEq":"sum","VtaNetaMeta":"sum","JEqMeta":"sum"}).reset_index()
dffy = df_prod1[(df_prod1["año"]=="2025")].groupby(["mes"]).agg(VtaAA=("VtaNeta","sum"), JEqAA = ("JEq","sum")).reset_index()
dffz = dffy.merge(dffx, on="mes", how="left")
dffz["Prod"] = dffz["VtaNeta"] / dffz["JEq"]
dffz["ProdMeta"] = dffz["VtaNetaMeta"] / dffz["JEqMeta"]
dffz["ProdAA"] = dffz["VtaAA"] / dffz["JEqAA"]
dffz["Cumplimiento_Meta"] = dffz["Prod"]/dffz["ProdMeta"]-1
dffz

,mes,VtaAA,JEqAA,VtaNeta,JEq,VtaNetaMeta,JEqMeta,Prod,ProdMeta,ProdAA,Cumplimiento_Meta
0,01,5.181167e+08,6171.840867,5.336714e+08,6049.577070,5.489281e+08,6153.103817,88216.315846,89211.576490,83948.493766,-0.011156
1,02,4.908670e+08,6276.841719,5.308168e+08,6167.054740,5.227860e+08,6153.103817,86072.980851,84962.970822,78202.857482,0.013065
2,03,6.027098e+08,6282.130430,6.441009e+08,6396.902816,6.411830e+08,6153.103817,100689.487577,104204.811729,95940.356729,-0.033735
3,04,5.062260e+08,6065.014861,5.556179e+08,5971.190313,5.430334e+08,6153.103817,93049.767708,88253.568051,83466.577397,0.054346
4,05,5.202420e+08,6132.419577,5.718481e+08,6049.699523,5.595939e+08,6153.103817,94525.042472,90944.976583,84834.710859,0.039365
5,06,4.785642e+08,6018.327729,5.204966e+08,6002.052292,5.223117e+08,6153.103817,86719.767751,84885.883123,79517.804489,0.021604
6,07,5.139323e+08,5996.786297,5.288369e+08,5889.773138,5.508907e+08,6153.103817,89789.006722,89530.534884,85701.293233,0.002887
7,08,5.005596e+08,5685.091351,NaN,NaN,NaN,NaN,NaN,NaN,88047.765815,NaN
8,09,4.604885e+08,5640.100132,NaN,NaN,NaN,NaN,NaN,NaN,81645.451449,NaN
9,10,4.753970e+08,5635.741163,NaN,NaN,NaN,NaN,NaN,NaN,84353.950116,NaN


## 2.2. Ranking GOS

In [8]:
df_gos = df_prod2.groupby("GOS").agg({"VtaNeta":"sum","JEq":"sum", "VtaNetaMeta":"sum","JEqMeta":"sum", "VtaNetaAA":"sum", "JEqAA":"sum"}).reset_index()
df_gos["Prod"] = df_gos["VtaNeta"] / df_gos["JEq"]
df_gos["ProdMeta"] = df_gos["VtaNetaMeta"] / df_gos["JEqMeta"]
df_gos["ProdAA"] = df_gos["VtaNetaAA"] / df_gos["JEqAA"]
df_gos["Cump_Meta"] = df_gos["Prod"]/df_gos["ProdMeta"]-1
df_gos["Cump_Meta2"] = df_gos["Prod"]/df_gos["ProdMeta"]
df_gos["Cump_AA"] = df_gos["Prod"]/df_gos["ProdAA"]-1
df_gos = df_gos.sort_values(by="Cump_Meta", ascending=False)
df_gos

,GOS,VtaNeta,JEq,VtaNetaMeta,JEqMeta,VtaNetaAA,JEqAA,Prod,ProdMeta,ProdAA,Cump_Meta,Cump_Meta2,Cump_AA
0,Fátima L,8.800525e+06,134.621526,8.628032e+06,142.260192,8.744653e+06,156.470276,65372.348697,60649.657302,55886.993616,0.077868,1.077868,0.169724
1,Hernán P,5.590972e+07,696.216754,5.367685e+07,708.059011,5.240952e+07,714.517802,80305.044050,75808.444179,73349.490219,0.059315,1.059315,0.094828
4,Sofía V,2.012756e+08,2064.648965,2.113451e+08,2229.046568,1.932782e+08,2044.204355,97486.599696,94814.123754,94549.360998,0.028186,1.028186,0.031066
3,Sebastián S,2.396639e+08,2590.728884,2.526115e+08,2682.391384,2.406545e+08,2719.583125,92508.271169,94173.988457,88489.472110,-0.017688,0.982312,0.045416
2,María R,2.318718e+07,403.557009,2.462920e+07,391.346660,1.884548e+07,362.010739,57457.012605,62934.482407,52057.793918,-0.087034,0.912966,0.103716


## 2.3. Ranking GOR

In [9]:
df_gor = df_prod2.groupby("GOR").agg({"VtaNeta":"sum","JEq":"sum", "VtaNetaMeta":"sum","JEqMeta":"sum", "VtaNetaAA":"sum", "JEqAA":"sum"}).reset_index()
df_gor["Prod"] = df_gor["VtaNeta"] / df_gor["JEq"]
df_gor["ProdMeta"] = df_gor["VtaNetaMeta"] / df_gor["JEqMeta"]
df_gor["ProdAA"] = df_gor["VtaNetaAA"] / df_gor["JEqAA"]
df_gor["Cump_Meta"] = df_gor["Prod"]/df_gor["ProdMeta"]-1
df_gor["Cump_Meta2"] = df_gor["Prod"]/df_gor["ProdMeta"]
df_gor["Cump_AA"] = df_gor["Prod"]/df_gor["ProdAA"]-1
df_gor = df_gor.sort_values(by="Cump_Meta", ascending=False)
df_gor

,GOR,VtaNeta,JEq,VtaNetaMeta,JEqMeta,VtaNetaAA,JEqAA,Prod,ProdMeta,ProdAA,Cump_Meta,Cump_Meta2,Cump_AA
1,Fátima L,8.800525e+06,134.621526,8.628032e+06,142.260192,8744653.29,156.470276,65372.348697,60649.657302,55886.993616,0.077868,1.077868,0.169724
6,Melany A,6.219145e+07,695.120276,6.732855e+07,800.715670,63762571.67,753.895013,89468.614752,84085.464548,84577.521450,0.064020,1.064020,0.057830
0,Daniel R,5.590972e+07,696.216754,5.367685e+07,708.059011,52409516.56,714.517802,80305.044050,75808.444179,73349.490219,0.059315,1.059315,0.094828
3,Johanna V,6.217757e+07,638.969973,6.389560e+07,689.166433,60809280.84,665.561552,97309.065566,92714.324185,91365.375026,0.049558,1.049558,0.065054
4,José A,6.976579e+07,674.178844,7.289020e+07,729.990429,62291585.32,574.447661,103482.616169,99850.904046,108437.355598,0.036371,1.036371,-0.045692
2,Joe H,6.931837e+07,695.349845,7.112634e+07,698.340470,67224058.51,715.861680,99688.479020,101850.525553,93906.491125,-0.021228,0.978772,0.061572
8,Vik E,9.503296e+07,1073.359819,1.013975e+08,1117.475898,97005565.68,1126.594442,88537.841301,90737.933940,86105.134240,-0.024247,0.975753,0.028253
7,Rodolfo O,8.245332e+07,878.399093,8.731844e+07,875.749054,82839628.57,927.427130,93867.717491,99707.148558,89321.981056,-0.058566,0.941434,0.050892
5,María R,2.318718e+07,403.557009,2.462920e+07,391.346660,18845480.46,362.010739,57457.012605,62934.482407,52057.793918,-0.087034,0.912966,0.103716


## 2.4. Ranking Tiendas

In [ ]:
df_tienda = df_prod2.groupby("Tienda").agg({"VtaNeta":"sum","JEq":"sum", "VtaNetaMeta":"sum","JEqMeta":"sum", "VtaNetaAA":"sum", "JEqAA":"sum"}).reset_index()
df_tienda["Prod"] = df_tienda["VtaNeta"] / df_tienda["JEq"]
df_tienda["ProdMeta"] = df_tienda["VtaNetaMeta"] / df_tienda["JEqMeta"]
df_tienda["ProdAA"] = df_tienda["VtaNetaAA"] / df_tienda["JEqAA"]
df_tienda["Cump_Meta"] = df_tienda["Prod"]/df_tienda["ProdMeta"]-1
df_tienda["Cump_Meta2"] = df_tienda["Prod"]/df_tienda["ProdMeta"]
df_tienda["Cump_AA"] = df_tienda["Prod"]/df_tienda["ProdAA"]-1
df_tienda = df_tienda.sort_values(by="Cump_Meta", ascending=False)
df_tienda1 = df_tienda[
    np.isfinite(df_tienda["Prod"]) &
    np.isfinite(df_tienda["ProdMeta"]) &
    (df_tienda["Prod"] > 1) &
    (df_tienda["ProdMeta"] > 1)
].copy()

regex_limpieza = r"^[A-Z0-9 ]+\s(?=[A-Z][a-z])|\s*-\s*[A-Z]+$"

df_tienda1["Tienda"] = df_tienda1["Tienda"].str.replace(
    regex_limpieza, "", regex=True
)
df_tienda1_topx = df_tienda1.sort_values(by="Cump_Meta", ascending=False)
# df_tienda1_top = df_tienda1_topx[~df_tienda1_topx["Tienda"].isin(["Primavera", "Asia"])]
df_tienda1_top = df_tienda1_topx.head(7)
df_tienda1_bot = df_tienda1.head(7).sort_values(by="Cump_Meta", ascending=True)


## 2.5. Heatmap GOR

In [11]:
df_prod3 = df_prod1[(df_prod1["año"]=="2026")&(df_prod1["Periodo"]<=periodo_analisis)]

df_gor_historico = df_prod3.groupby(["GOR", "mes"]).agg({"VtaNeta":"sum","JEq":"sum", "VtaNetaMeta":"sum","JEqMeta":"sum", "VtaNetaAA":"sum", "JEqAA":"sum"}).reset_index()
df_gor_historico["Prod"] = df_gor_historico["VtaNeta"] / df_gor_historico["JEq"]
df_gor_historico["ProdMeta"] = df_gor_historico["VtaNetaMeta"] / df_gor_historico["JEqMeta"]
df_gor_historico["ProdAA"] = df_gor_historico["VtaNetaAA"] / df_gor_historico["JEqAA"]
df_gor_historico["Cump_Meta"] = df_gor_historico["Prod"]/df_gor_historico["ProdMeta"]-1
df_gor_historico["Cump_Meta2"] = df_gor_historico["Prod"]/df_gor_historico["ProdMeta"]
df_gor_historico["Cump_AA"] = df_gor_historico["Prod"]/df_gor_historico["ProdAA"]-1
df_gor_historico = df_gor_historico.sort_values(by="Cump_Meta", ascending=False)


## 2.6. Ranking tiendas GOR

In [12]:
df_gor_bot = df_prod1[(df_prod1["Periodo"]==periodo_analisis)]
df_gor_bot["Prod"] = df_gor_bot["VtaNeta"] / df_gor_bot["JEq"]
df_gor_bot["ProdMeta"] = df_gor_bot["VtaNetaMeta"] / df_gor_bot["JEqMeta"]
df_gor_bot["ProdAA"] = df_gor_bot["VtaNetaAA"] / df_gor_bot["JEqAA"]
df_gor_bot["Cump_Meta"] = df_gor_bot["Prod"]/df_gor_bot["ProdMeta"]-1
df_gor_bot["Cump_AA"] = df_gor_bot["Prod"]/df_gor_bot["ProdAA"]-1
df_gor_bot["Tienda"] = df_gor_bot["Tienda"].str.replace(
    regex_limpieza, "", regex=True
)

lista_gor = df_gor_bot["GOR"].unique().tolist()


## 2.7. Dashboard

In [ ]:
graff.KPI_Productividad_HTML(
    prod,
    "PRODUCTIVIDAD",
    prod_meta,
    prod_aa,
    var_meta,
    var_aa,
    r"html/kpi_productividad.html"
)

graff.Historico_Comparativo(
    df=dffz,
    anio1="ProdAA",
    anio2="Prod",
    cumplimiento="Cumplimiento_Meta",
    titulo = "Productividad -  Histórico",
    archivo_html=r"html/Historico_Productividad.html",
    ymin=None,
    ymax=None,
    y2min=-0.4,
    y2max=0.10,
    col_mes="mes"
)

graff.Ranking_HTML(
    df=df_gos,
    col_nombre="GOS",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Prod",
    titulo="Ranking GOS (▲ Meta | ▲ AA | Prod)",
    archivo_html=r"html/Ranking_GOS.html"
)

graff.Ranking_HTML(
    df=df_gor,
    col_nombre="GOR",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Prod",
    titulo="Ranking GOR (▲ Meta | ▲ AA | Prod)",
    archivo_html=r"html/Ranking_GOR.html"
)

graff.Ranking_HTML(
    df=df_tienda1_top,
    col_nombre="Tienda",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Prod",
    titulo="Tiendas TOP (▲ Meta | ▲ AA | Prod)",
    archivo_html=r"html/Ranking_TiendasTOP.html"
)

graff.Ranking_HTML(
    df=df_tienda1_bot,
    col_nombre="Tienda",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Prod",
    titulo="Tiendas BOT (▲ Meta | ▲ AA | Prod)",
    archivo_html=r"html/Ranking_TiendasBOT.html"
)



In [14]:
graff.generar_reporte_consolidado(
    kpi_path="kpi_productividad.html",
    historico_path="Historico_Productividad.html",
    gos_path="Ranking_GOS.html",
    gor_path="Ranking_GOR.html",
    top_path="Ranking_TiendasTOP.html",
    bot_path="Ranking_TiendasBOT.html",
    output_path=r"html/dashboard_consolidado.html"
)
graff.HTML_a_PNG(
    r"html/dashboard_consolidado.html",
    r"png/01_Dashboard_Productividad4.png")

'png/01_Dashboard_Productividad4.png'

In [ ]:
graff.Heatmap_GOR(
    df_gor_historico,
    gor="GOR",
    mes="mes",
    valor="Cump_Meta",
    archivo_html=r"html/Heatmap_GOR.html",
    titulo="Cumplimiento vs Meta")
graff.HTML_a_PNG( r"html/Heatmap_GOR.html", r"png/02_Dashboard_historico_GOR.png")

graff.tiendas_top_bot_gor(df_gor_bot, lista_gor, r"html/productividad_tiendas_bot_gor.html", ascending=True)
graff.HTML_a_PNG( r"html/productividad_tiendas_bot_gor.html", r"png/03_Dashboard_productividad_tiendas_bot_gor.png")

graff.tiendas_top_bot_gor(df_gor_bot, lista_gor, r"html/productividad_tiendas_top_gor.html", ascending=False)
graff.HTML_a_PNG(r"html/productividad_tiendas_top_gor.html", r"png/04_Dashboard_productividad_tiendas_top_gor.png")

Archivo generado: html/Heatmap_GOR.html


'png/04_Dashboard_productividad_tiendas_top_gor.png'

# 3. Velocidad

In [ ]:
df_veloc = pd.read_excel(ruta_velocdidad)

df_veloc["GOS"] = ( df_veloc["regional"].str.split().str[0]  + " " + df_veloc["regional"].str.split().str[1].str[0])
df_veloc["GOR"] = (df_veloc["supervisor"].str.title().str.split().str[0] + " " + df_veloc["supervisor"].str.title().str.split().str[1].str[0])

meses_dict = {
    "enero": "01",
    "febrero": "02",
    "marzo": "03",
    "abril": "04",
    "mayo": "05",
    "junio": "06",
    "julio": "07",
    "agosto": "08",
    "septiembre": "09",
    "octubre": "10",
    "noviembre": "11",
    "diciembre": "12"
}

df_veloc = df_veloc.dropna(subset=["Date - Año"])

df_veloc["mes"] = df_veloc["Date - Mes"].str.lower().map(meses_dict)
df_veloc["año"] = df_veloc["Date - Año"].astype(int).astype(str)
df_veloc["Periodo"] = df_veloc["año"].astype(str) + df_veloc["mes"]
df_veloc["Tienda"] = df_veloc["nombre"].str.split(" - ").str[0]

df_aa = ( df_veloc[["Tienda", "mes", "año", "venta_unidad", "tiempo_escaneo"]].rename(columns={"venta_unidad": "venta_unidad_aa", "tiempo_escaneo": "tiempo_escaneo_aa"}))
df_aa["año"] = df_aa["año"].astype(int)
df_aa["año"] = df_aa["año"] + 1
df_aa["año"] = df_aa["año"].astype(str)

df_veloc1 = df_veloc.merge( df_aa, on=["Tienda", "mes", "año"], how="left")

df_veloc2 = df_veloc1[df_veloc1["Periodo"]==periodo_analisis]
df_veloc2

venta_unid = df_veloc2["venta_unidad"].sum()
tiempo_scan = df_veloc2["tiempo_escaneo"].sum()
venta_unid_aa = df_veloc2["venta_unidad_aa"].sum()
tiempo_scan_aa = df_veloc2["tiempo_escaneo_aa"].sum()


velocidad = venta_unid/tiempo_scan*60
velocidad_aa = venta_unid_aa/tiempo_scan_aa*60
velocidad_meta = df_veloc2["Objetivo"].mean()

var_veloc_meta = velocidad/velocidad_meta
var_veloc_aa = velocidad/velocidad_aa

## 3.1. Historico

In [20]:
df_veloc1x = df_veloc1[(df_veloc1["año"]=="2026")&(df_veloc1["Periodo"]<=periodo_analisis)].groupby(["mes"]).agg({"venta_unidad":"sum", "tiempo_escaneo":"sum","Objetivo":"mean"}).reset_index()
df_veloc1y = df_veloc1[(df_veloc1["año"]=="2025")].groupby(["mes"]).agg(venta_unidad_AA=("venta_unidad","sum"), tiempo_escaneo_AA = ("tiempo_escaneo","sum")).reset_index()
df_veloc1z = df_veloc1y.merge(df_veloc1x, on="mes", how="left")
df_veloc1z["Velocidad"] = df_veloc1z["venta_unidad"] / df_veloc1z["tiempo_escaneo"]*60
df_veloc1z["VelocidadMeta"] = df_veloc1z["Objetivo"]
df_veloc1z["VelocidadAA"] = df_veloc1z["venta_unidad_AA"] / df_veloc1z["tiempo_escaneo_AA"]*60
df_veloc1z["Cumplimiento_Meta"] = df_veloc1z["Velocidad"]/df_veloc1z["VelocidadMeta"]-1
df_veloc1z

,mes,venta_unidad_AA,tiempo_escaneo_AA,venta_unidad,tiempo_escaneo,Objetivo,Velocidad,VelocidadMeta,VelocidadAA,Cumplimiento_Meta
0,01,5.327118e+07,263212356.0,51071758.59,260697425.0,13.814159,11.754261,13.814159,12.143315,-0.149115
1,02,4.983687e+07,243574678.0,49019695.06,237332596.0,13.814159,12.392658,13.814159,12.276368,-0.102902
2,03,5.962989e+07,284186136.0,58228078.64,269503570.0,13.814159,12.963408,13.814159,12.589612,-0.061585
3,04,5.156991e+07,248547228.0,48629474.45,222641363.0,13.814159,13.105240,13.814159,12.449120,-0.051318
4,05,5.301125e+07,254613326.0,52318193.50,238340352.0,13.814159,13.170626,13.814159,12.492178,-0.046585
5,06,5.018963e+07,246690384.0,48305160.50,219543555.0,13.814159,13.201525,13.814159,12.207115,-0.044348
6,07,5.154568e+07,243262231.0,20046370.84,90722134.0,13.814159,13.257870,13.814159,12.713608,-0.040270
7,08,5.125575e+07,242908280.0,NaN,NaN,NaN,NaN,NaN,12.660520,NaN
8,09,4.811315e+07,226561861.0,NaN,NaN,NaN,NaN,NaN,12.741726,NaN
9,10,4.989798e+07,230922315.0,NaN,NaN,NaN,NaN,NaN,12.964874,NaN


## 3.2. Ranking GOS

In [21]:
df_vel_gos = df_veloc2.groupby("GOS").agg({"venta_unidad":"sum","tiempo_escaneo":"sum", "Objetivo":"mean", "venta_unidad_aa":"sum", "tiempo_escaneo_aa":"sum"}).reset_index()
df_vel_gos["Velocidad"] = df_vel_gos["venta_unidad"] / df_vel_gos["tiempo_escaneo"]*60
df_vel_gos["VelocidadMeta"] = df_vel_gos["Objetivo"]
df_vel_gos["VelocidadAA"] = df_vel_gos["venta_unidad_aa"] / df_vel_gos["tiempo_escaneo_aa"]*60
df_vel_gos["Cump_Meta"] = df_vel_gos["Velocidad"]/df_vel_gos["VelocidadMeta"]-1
df_vel_gos["Cump_Meta2"] = df_vel_gos["Velocidad"]/df_vel_gos["VelocidadMeta"]
df_vel_gos["Cump_AA"] = df_vel_gos["Velocidad"]/df_vel_gos["VelocidadAA"]-1
df_vel_gos = df_vel_gos.sort_values(by="Cump_Meta", ascending=False)
df_vel_gos

,GOS,venta_unidad,tiempo_escaneo,Objetivo,venta_unidad_aa,tiempo_escaneo_aa,Velocidad,VelocidadMeta,VelocidadAA,Cump_Meta,Cump_Meta2,Cump_AA
2,María R,745321.01,4771765.0,9.142857,1840448.33,12466435.0,9.371639,9.142857,8.857937,0.025023,1.025023,0.057993
0,Fátima L,507838.91,2254898.0,13.857143,1202124.00,6336630.0,13.512955,13.857143,11.382618,-0.024838,0.975162,0.187157
1,Hernán P,2456471.59,11581961.0,13.125000,5772422.21,30694818.0,12.725677,13.125000,11.283512,-0.030425,0.969575,0.127812
4,Sofía V,7281375.58,30500789.0,14.914286,19021981.57,80077539.0,14.323647,14.914286,14.252672,-0.039602,0.960398,0.004980
3,Sebastián S,9055363.75,41612721.0,13.926829,23708702.76,113686809.0,13.056628,13.926829,12.512640,-0.062484,0.937516,0.043475


## 3.3. Ranking GOR

In [22]:
df_vel_gor = df_veloc2.groupby("GOR").agg({"venta_unidad":"sum","tiempo_escaneo":"sum", "Objetivo":"mean", "venta_unidad_aa":"sum", "tiempo_escaneo_aa":"sum"}).reset_index()
df_vel_gor["Velocidad"] = df_vel_gor["venta_unidad"] / df_vel_gor["tiempo_escaneo"]*60
df_vel_gor["VelocidadMeta"] = df_vel_gor["Objetivo"]
df_vel_gor["VelocidadAA"] = df_vel_gor["venta_unidad_aa"] / df_vel_gor["tiempo_escaneo_aa"]*60
df_vel_gor["Cump_Meta"] = df_vel_gor["Velocidad"]/df_vel_gor["VelocidadMeta"]-1
df_vel_gor["Cump_Meta2"] = df_vel_gor["Velocidad"]/df_vel_gor["VelocidadMeta"]
df_vel_gor["Cump_AA"] = df_vel_gor["Velocidad"]/df_vel_gor["VelocidadAA"]-1
df_vel_gor = df_vel_gor.sort_values(by="Cump_Meta", ascending=False)
df_vel_gor

,GOR,venta_unidad,tiempo_escaneo,Objetivo,venta_unidad_aa,tiempo_escaneo_aa,Velocidad,VelocidadMeta,VelocidadAA,Cump_Meta,Cump_Meta2,Cump_AA
4,María R,745321.01,4771765.0,9.142857,1840448.33,12466435.0,9.371639,9.142857,8.857937,0.025023,1.025023,0.057993
6,Pepe A,2401940.64,9695346.0,14.909091,6320454.13,25158590.0,14.864497,14.909091,15.073470,-0.002991,0.997009,-0.013864
1,Fátima L,507838.91,2254898.0,13.857143,1202124.00,6336630.0,13.512955,13.857143,11.382618,-0.024838,0.975162,0.187157
0,Daniel R,2456471.59,11581961.0,13.125000,5772422.21,30694818.0,12.725677,13.125000,11.283512,-0.030425,0.969575,0.127812
8,Vik E,3869085.37,17056497.0,14.166667,10085700.07,46569205.0,13.610363,14.166667,12.994467,-0.039268,0.960732,0.047397
7,Rodolfo O,2983760.57,13385301.0,14.066667,8000192.53,36746326.0,13.374793,14.066667,13.062845,-0.049185,0.950815,0.023881
2,Joe H,2533847.73,10916440.0,14.727273,6182477.73,27648927.0,13.926781,14.727273,13.416386,-0.054354,0.945646,0.038043
5,Melany A,2345587.21,9889003.0,15.076923,6519049.71,27270022.0,14.231489,15.076923,14.343332,-0.056075,0.943925,-0.007798
3,Johanna V,2202517.81,11170923.0,13.125000,5622810.16,30371278.0,11.829915,13.125000,11.108147,-0.098673,0.901327,0.064976


## 3.4. Ranking Tiendas

In [23]:
df_veloc_tienda = df_veloc2.groupby("Tienda").agg({"venta_unidad":"sum","tiempo_escaneo":"sum", "Objetivo":"mean", "venta_unidad_aa":"sum", "tiempo_escaneo_aa":"sum"}).reset_index()
df_veloc_tienda["Velocidad"] = df_veloc_tienda["venta_unidad"] / df_veloc_tienda["tiempo_escaneo"]*60
df_veloc_tienda["VelocidadMeta"] = df_veloc_tienda["Objetivo"]
df_veloc_tienda["VelocidadAA"] = df_veloc_tienda["venta_unidad_aa"] / df_veloc_tienda["tiempo_escaneo_aa"]*60
df_veloc_tienda["Cump_Meta"] = df_veloc_tienda["Velocidad"]/df_veloc_tienda["VelocidadMeta"]-1
df_veloc_tienda["Cump_Meta2"] = df_veloc_tienda["Velocidad"]/df_veloc_tienda["VelocidadMeta"]
df_veloc_tienda["Cump_AA"] = df_veloc_tienda["Velocidad"]/df_veloc_tienda["VelocidadAA"]-1
df_veloc_tienda = df_veloc_tienda.sort_values(by="Cump_Meta", ascending=False)

df_veloc_tienda_top = df_veloc_tienda.sort_values(by="Cump_Meta", ascending=False)
df_veloc_tienda_bot = df_veloc_tienda.sort_values(by="Cump_Meta", ascending=True)

## 3.5. Heatmap GOR

In [24]:
df_veloc3 = df_veloc1[(df_veloc1["año"]=="2026")&(df_veloc1["Periodo"]<=periodo_analisis)]

df_gor_veloc_hist = df_veloc3.groupby(["GOR", "mes"]).agg({"venta_unidad":"sum","tiempo_escaneo":"sum", "Objetivo":"mean", "venta_unidad_aa":"sum", "tiempo_escaneo_aa":"sum"}).reset_index()
df_gor_veloc_hist["Velocidad"] = df_gor_veloc_hist["venta_unidad"] / df_gor_veloc_hist["tiempo_escaneo"]*60
df_gor_veloc_hist["VelocidadMeta"] = df_gor_veloc_hist["Objetivo"]
df_gor_veloc_hist["VelocidadAA"] = df_gor_veloc_hist["venta_unidad_aa"] / df_gor_veloc_hist["tiempo_escaneo_aa"]*60
df_gor_veloc_hist["Cump_Meta"] = df_gor_veloc_hist["Velocidad"]/df_gor_veloc_hist["VelocidadMeta"]-1
df_gor_veloc_hist["Cump_Meta2"] = df_gor_veloc_hist["Velocidad"]/df_gor_veloc_hist["VelocidadMeta"]
df_gor_veloc_hist["Cump_AA"] = df_gor_veloc_hist["Velocidad"]/df_gor_veloc_hist["VelocidadAA"]-1
df_gor_veloc_hist = df_gor_veloc_hist.sort_values(by="Cump_Meta", ascending=False)
df_gor_veloc_hist.head()

,GOR,mes,venta_unidad,tiempo_escaneo,Objetivo,venta_unidad_aa,tiempo_escaneo_aa,Velocidad,VelocidadMeta,VelocidadAA,Cump_Meta,Cump_Meta2,Cump_AA
34,María R,07,745321.01,4771765.0,9.142857,1840448.330,12466435.0,9.371639,9.142857,8.857937,0.025023,1.025023,0.057993
33,María R,06,1835060.89,11763333.0,9.142857,1992441.470,13873864.0,9.359903,9.142857,8.616669,0.023739,1.023739,0.086255
32,María R,05,1951870.94,12749904.0,9.142857,2048567.460,14139822.0,9.185344,9.142857,8.692758,0.004647,1.004647,0.056666
48,Pepe A,07,2401940.64,9695346.0,14.909091,6320454.130,25158590.0,14.864497,14.909091,15.073470,-0.002991,0.997009,-0.013864
29,María R,02,1744071.01,11593894.0,9.142857,2102530.538,14178704.0,9.025808,9.142857,8.897275,-0.012802,0.987198,0.014446


## 3.6. Ranking tiendas GOR

In [25]:
df_veloc_gor_bot = df_veloc1[(df_veloc1["Periodo"]==periodo_analisis)]
df_veloc_gor_bot["Velocidad"] = df_veloc_gor_bot["venta_unidad"] / df_veloc_gor_bot["tiempo_escaneo"]*60
df_veloc_gor_bot["VelocidadMeta"] = df_veloc_gor_bot["Objetivo"]
df_veloc_gor_bot["VelocidadAA"] = df_veloc_gor_bot["venta_unidad_aa"] / df_veloc_gor_bot["tiempo_escaneo_aa"]*60
df_veloc_gor_bot["Cump_Meta"] = df_veloc_gor_bot["Velocidad"]/df_veloc_gor_bot["VelocidadMeta"]-1
df_veloc_gor_bot["Cump_AA"] = df_veloc_gor_bot["Velocidad"]/df_veloc_gor_bot["VelocidadAA"]-1

lista_gor = df_veloc_gor_bot["GOR"].unique().tolist()


## 3.7. Dashboard

In [26]:
graff.KPI_Productividad_HTML(
    velocidad,
    "VELOCIDAD",
    velocidad_meta,
    velocidad_aa,
    var_veloc_meta,
    var_veloc_aa,
    r"html/kpi_velocidad.html"
)

graff.Historico_Comparativo(
    df=df_veloc1z,
    anio1="VelocidadAA",
    anio2="Velocidad",
    cumplimiento="Cumplimiento_Meta",
    titulo = "Velocidad -  Histórico",
    archivo_html=r"html/Historico_Velocidad.html",
    ymin=None,
    ymax=None,
    y2min=-0.8,
    y2max=0.10,
    col_mes="mes"
)

graff.Ranking_HTML(
    df=df_vel_gos,
    col_nombre="GOS",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Velocidad",
    titulo="Ranking GOS (▲ Meta | ▲ AA | Veloc)",
    archivo_html=r"html/Ranking_Velocidad_GOS.html"
)

graff.Ranking_HTML(
    df=df_vel_gor,
    col_nombre="GOR",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Velocidad",
    titulo="Ranking GOR (▲ Meta | ▲ AA | Veloc)",
    archivo_html=r"html/Ranking_Velocidad_GOR.html"
)

graff.Ranking_HTML(
    df=df_veloc_tienda_top.head(7),
    col_nombre="Tienda",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Velocidad",
    titulo="Tiendas TOP (▲ Meta | ▲ AA | Veloc)",
    archivo_html=r"html/Ranking_Velocidad_TiendasTOP.html"
)

graff.Ranking_HTML(
    df=df_veloc_tienda_bot.head(7),
    col_nombre="Tienda",
    col_cumplimiento="Cump_Meta2",
    col_valor1="Cump_Meta",
    col_valor2="Cump_AA",
    col_valor3="Velocidad",
    titulo="Tiendas BOT (▲ Meta | ▲ AA | Veloc)",
    archivo_html=r"html/Ranking_Velocidad_TiendasBOT.html"
)

graff.generar_reporte_consolidado(
    kpi_path=r"kpi_velocidad.html",
    historico_path=r"Historico_Velocidad.html",
    gos_path=r"Ranking_Velocidad_GOS.html",
    gor_path=r"Ranking_Velocidad_GOR.html",
    top_path=r"Ranking_Velocidad_TiendasTOP.html",
    bot_path=r"Ranking_Velocidad_TiendasBOT.html",
    output_path=r"html/dashboard_velocidad_consolidado.html")
graff.HTML_a_PNG(
    r"html/dashboard_velocidad_consolidado.html",
    r"png/05_Dashboard_Velocidad.png")

graff.Heatmap_GOR(
    df_gor_veloc_hist,
    gor="GOR",
    mes="mes",
    valor="Cump_Meta",
    archivo_html=r"html/Heatmap_Velocidad_GOR.html",
    titulo="Cumplimiento vs Meta")
graff.HTML_a_PNG( r"html/Heatmap_Velocidad_GOR.html", r"png/06_Heatmap_Velocidad_GOR.png")

graff.tiendas_top_bot_gor(df_veloc_gor_bot, lista_gor, r"html/velocidad_tiendas_bot_gor.html", ascending=True)
graff.HTML_a_PNG( r"html/velocidad_tiendas_bot_gor.html", r"png/07_velocidad_tiendas_bot_gor.png")

graff.tiendas_top_bot_gor(df_veloc_gor_bot, lista_gor, r"html/velocidad_tiendas_top_gor.html", ascending=False)
graff.HTML_a_PNG(r"html/velocidad_tiendas_top_gor.html", r"png/08_velocidad_tiendas_top_gor.png")

Archivo generado: html/Heatmap_Velocidad_GOR.html


'png/08_velocidad_tiendas_top_gor.png'

#4. 